<a href="https://colab.research.google.com/github/micaelaruggeri/educacion-pba/blob/main/M%C3%A1s_docentes%2C_Menos_escuelas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Más Docentes, Menos Escuelas
## Análisis de Infraestructura y Matrícula Educativa en la Provincia de Buenos Aires (2015–2025)
**Prácticas Profesionalizantes — Proyecto Integrador (TP N.º 2, N.º 3 y N.º 4)**  
**Autora:** Lic. Micaela Leslie Ruggeri  

---

## Etapa 1: Relevamiento, Calidad y Preparación de Datos (TP N.º 2)

### 1.1 Contexto y Fuente de Datos
Se utiliza el dataset público **"Unidades de servicio y matrícula - municipios (2015 - 2025)"**, publicado por la Dirección General de Cultura y Educación (DGCyE) del Gobierno de la Provincia de Buenos Aires. Contiene **1.485 registros** correspondientes a los **135 municipios bonaerenses** a lo largo de **11 años consecutivos (2015–2025)**.

### 1.2 Carga y Exploración Inicial
*(Nota: Reemplazar la ruta del archivo por el enlace RAW del repositorio público de GitHub para ejecución automática).*

In [11]:
import pandas as pd
import numpy as np

# Carga del dataset original (reemplazar por URL Raw de GitHub si se ejecuta online)
url_original = "/evolucion-unidades-servicio-y-matricula-municipios-2015_2025.csv"
df = pd.read_csv(url_original)

print("=== Estructura inicial del dataset ===")
print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"Valores nulos totales: {df.isnull().sum().sum()}")
print(f"Filas duplicadas: {df.duplicated().sum()}\n")
df.info()
display(df.head())

=== Estructura inicial del dataset ===
Dimensiones: 1485 filas x 5 columnas
Valores nulos totales: 0
Filas duplicadas: 0

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1485 entries, 0 to 1484
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   anio                  1485 non-null   int64 
 1   municipio_nombre      1485 non-null   object
 2   unidades_de_servicio  1485 non-null   int64 
 3   matricula             1485 non-null   int64 
 4   docentes              1485 non-null   int64 
dtypes: int64(4), object(1)
memory usage: 58.1+ KB


,anio,municipio_nombre,unidades_de_servicio,matricula,docentes
0,2015,Adolfo Alsina,83,5500,793
1,2015,Alberti,42,4055,521
2,2015,Almirante Brown,546,152622,14325
3,2015,Arrecifes,65,9225,996
4,2015,Avellaneda,439,116019,11424


### 1.3 Limpieza, Transformación e Ingeniería de Características
Tras evaluar la calidad del conjunto de datos, se aplicaron las siguientes decisiones:
1. **Renombrado de columna:** Se simplifica `municipio_nombre` a `municipio`.
2. **Normalización de texto:** Se convierten los nombres de los municipios a minúsculas, eliminando tildes y espacios residuales para evitar categorías duplicadas al agrupar o cruzar datos.
3. **Creación de métricas relativas:** Dado que los totales absolutos impiden comparar equitativamente distritos densamente poblados con municipios pequeños, se construyen dos indicadores clave:
   * `alumnos_por_docente`: estima la carga áulica y presión sobre el recurso humano.
   * `alumnos_por_escuela`: estima la presión sobre la infraestructura educativa.

In [12]:
# Renombrar columna
df = df.rename(columns={'municipio_nombre': 'municipio'})

# Normalización de cadenas de texto en 'municipio'
df['municipio'] = (
    df['municipio']
    .str.lower()
    .str.normalize('NFKD')
    .str.encode('ascii', errors='ignore')
    .str.decode('utf-8')
    .str.strip()
)

# Creación de métricas de proporción
df['alumnos_por_docente'] = round(df['matricula'] / df['docentes'], 2)
df['alumnos_por_escuela'] = round(df['matricula'] / df['unidades_de_servicio'], 2)

# Validación post-limpieza
print("=== Validación post limpieza ===")
print(f"Cantidad de registros final: {df.shape[0]}")
print(f"Cantidad de columnas final: {df.shape[1]}")
print(f"Nulos encontrados: {df.isnull().sum().sum()}")
print(f"Duplicados encontrados: {df.duplicated().sum()}")
display(df.head())

# Exportación del dataset limpio
df.to_csv('dataset_limpio.csv', index=False)
print("\nArchivo 'dataset_limpio.csv' generado con éxito.")

=== Validación post limpieza ===
Cantidad de registros final: 1485
Cantidad de columnas final: 7
Nulos encontrados: 0
Duplicados encontrados: 0


,anio,municipio,unidades_de_servicio,matricula,docentes,alumnos_por_docente,alumnos_por_escuela
0,2015,adolfo alsina,83,5500,793,6.94,66.27
1,2015,alberti,42,4055,521,7.78,96.55
2,2015,almirante brown,546,152622,14325,10.65,279.53
3,2015,arrecifes,65,9225,996,9.26,141.92
4,2015,avellaneda,439,116019,11424,10.16,264.28



Archivo 'dataset_limpio.csv' generado con éxito.


---

## Etapa 2: Exploración y Análisis de Datos (TP N.º 3)

### 2.1 Estadística Descriptiva General
Observamos la distribución global de las variables numéricas a lo largo de toda la serie histórica (2015–2025).

In [13]:
display(df.describe().round(2))

,anio,unidades_de_servicio,matricula,docentes,alumnos_por_docente,alumnos_por_escuela
count,1485.00,1485.00,1485.00,1485.00,1485.00,1485.00
mean,2020.00,160.24,37633.79,4036.47,8.62,174.31
std,3.16,185.88,59144.79,6009.80,1.62,84.99
min,2015.00,13.00,810.00,101.00,1.28,38.52
25%,2017.00,58.00,5981.00,803.00,7.54,109.78
50%,2020.00,87.00,12888.00,1587.00,8.47,144.45
75%,2023.00,178.00,40496.00,4371.00,9.58,232.85
max,2025.00,1321.00,447867.00,46940.00,15.26,442.24


### 2.2 Pregunta 1: ¿Cómo evolucionó la matrícula y la asignación de cargos docentes en General Pueyrredon (2015–2025)?
* **Hallazgo:** Entre 2015 y 2025, la matrícula del distrito descendió levemente (de 196.001 a 188.140 alumnos), mientras que los cargos docentes crecieron sostenidamente (de 19.473 a 22.695). Esto permitió una mejora sustancial en la carga áulica local, bajando de **10,07 a 8,29 alumnos por docente**.

In [14]:
gp = df[df['municipio'] == 'general pueyrredon'].sort_values('anio')
display(gp[['anio', 'unidades_de_servicio', 'matricula', 'docentes', 'alumnos_por_docente', 'alumnos_por_escuela']])

,anio,unidades_de_servicio,matricula,docentes,alumnos_por_docente,alumnos_por_escuela
50,2015,801,196001,19473,10.07,244.70
185,2016,794,196973,19991,9.85,248.08
320,2017,773,197608,20049,9.86,255.64
455,2018,768,196922,20698,9.51,256.41
590,2019,763,200750,22644,8.87,263.11
725,2020,741,194596,22230,8.75,262.61
860,2021,748,197715,22137,8.93,264.32
995,2022,742,195232,21205,9.21,263.12
1130,2023,744,194949,23640,8.25,262.03
1265,2024,737,187309,24104,7.77,254.15


### 2.3 Pregunta 2: ¿Existe correlación lineal entre matrícula y creación de cargos docentes a nivel provincial?
* **Hallazgo:** El coeficiente de correlación de Pearson arroja un valor de **0,9874**, confirmando que la Provincia asigna cargos docentes en estricta proporción lineal respecto al volumen de estudiantes inscriptos.

In [15]:
corr = df['matricula'].corr(df['docentes'])
print(f"Coeficiente de correlación de Pearson (Matrícula vs. Docentes): {round(corr, 4)}")

Coeficiente de correlación de Pearson (Matrícula vs. Docentes): 0.9875


### 2.4 Pregunta 3: ¿Cuáles son los municipios con aulas más pobladas (mayor ratio alumnos/docente) en 2025?
* **Hallazgo:** Los distritos con mayor presión sobre el recurso humano pertenecen al conurbano periférico, encabezados por **Presidente Perón (12,52)**, **General Rodríguez (12,32)** y **Ezeiza (11,67)**. En el extremo opuesto se ubican municipios rurales del interior con baja densidad poblacional.

In [16]:
df_2025 = df[df['anio'] == 2025].copy()

top_ratio = df_2025.sort_values('alumnos_por_docente', ascending=False).head(5)
bottom_ratio = df_2025.sort_values('alumnos_por_docente', ascending=True).head(5)

print("Top 5 Municipios más saturados (2025):")
display(top_ratio[['municipio', 'matricula', 'docentes', 'alumnos_por_docente']])

print("\nTop 5 Municipios menos saturados (2025):")
display(bottom_ratio[['municipio', 'matricula', 'docentes', 'alumnos_por_docente']])

Top 5 Municipios más saturados (2025):


,municipio,matricula,docentes,alumnos_por_docente
1448,presidente peron,33957,2713,12.52
1401,general rodriguez,52108,4228,12.32
1386,ezeiza,66924,5734,11.67
1410,jose c. paz,90494,7951,11.38
1387,florencio varela,136715,12129,11.27



Top 5 Municipios menos saturados (2025):


,municipio,matricula,docentes,alumnos_por_docente
1380,daireaux,5940,4640,1.28
1397,general madariaga,10947,2997,3.65
1457,saavedra,7882,1882,4.19
1420,lezama,2365,468,5.05
1412,juarez,8959,1705,5.25


### 2.5 Pregunta 4: ¿Cómo varió el promedio municipal de alumnos por escuela a lo largo de la década?
* **Hallazgo:** El promedio simple municipal de alumnos por unidad de servicio alcanzó su pico en **2021 (179,5)** durante la pandemia, descendiendo de manera sostenida hasta **173,5 en 2025**.

In [17]:
temporal_escuela = df.groupby('anio')['alumnos_por_escuela'].mean().round(1)
display(temporal_escuela.to_frame(name='promedio_alumnos_por_escuela'))

,promedio_alumnos_por_escuela
anio,
2015,164.7
2016,165.7
2017,170.4
2018,174.4
2019,178.6
2020,179.0
2021,179.5
2022,178.5
2023,177.5


### 2.6 Pregunta 5: ¿Existen diferencias significativas entre el Conurbano bonaerense y el Interior en 2025?
* **Hallazgo:** Se confirma una brecha territorial marcada. En 2025, los 24 partidos del Conurbano promedian **8,98 alumnos por docente** frente a **7,54** en el Interior provincial (una presión casi 20% superior sobre el recurso docente en el GBA).

In [18]:
gba = [
    'almirante brown', 'avellaneda', 'berazategui', 'esteban echeverria', 'ezeiza',
    'florencio varela', 'general san martin', 'hurlingham', 'ituzaingo', 'jose c. paz',
    'la matanza', 'lanus', 'lomas de zamora', 'malvinas argentinas', 'merlo', 'moreno',
    'moron', 'quilmes', 'san fernando', 'san isidro', 'san miguel', 'tigre',
    'tres de febrero', 'vicente lopez'
]

df_2025['zona'] = np.where(df_2025['municipio'].isin(gba), 'Conurbano', 'Interior')

comp_zona = df_2025.groupby('zona')[['alumnos_por_docente', 'alumnos_por_escuela']].mean().round(2)
display(comp_zona)

,alumnos_por_docente,alumnos_por_escuela
zona,,
Conurbano,8.98,285.03
Interior,7.54,149.38


### 2.7 Pregunta 6 y Detección de Anomalías: Evolución de la infraestructura educativa (2015–2025)
* **Hallazgo:** A pesar de que los cargos docentes aumentaron un 20,2% en la década, la cantidad de unidades de servicio registró una **caída neta de 618 unidades** a nivel provincial (-2,79%) y de **59 unidades** en General Pueyrredon (-7,37%). Esto sugiere procesos de fusión de establecimientos, cierre de anexos o reorganizaciones administrativas.

In [19]:
total_2015 = df[df['anio'] == 2015]['unidades_de_servicio'].sum()
total_2025 = df[df['anio'] == 2025]['unidades_de_servicio'].sum()
var_prov = total_2025 - total_2015
porc_prov = round(var_prov / total_2015 * 100, 2)

gp_2015 = df[(df['anio'] == 2015) & (df['municipio'] == 'general pueyrredon')]['unidades_de_servicio'].sum()
gp_2025 = df[(df['anio'] == 2025) & (df['municipio'] == 'general pueyrredon')]['unidades_de_servicio'].sum()
var_gp = gp_2025 - gp_2015
porc_gp = round(var_gp / gp_2015 * 100, 2)

print(f"Provincia - 2015: {total_2015} | 2025: {total_2025} | Variación absoluta: {var_prov} ({porc_prov}%)")
print(f"Gral. Pueyrredon - 2015: {gp_2015} | 2025: {gp_2025} | Variación absoluta: {var_gp} ({porc_gp}%)")

Provincia - 2015: 22137 | 2025: 21519 | Variación absoluta: -618 (-2.79%)
Gral. Pueyrredon - 2015: 801 | 2025: 742 | Variación absoluta: -59 (-7.37%)


---

## Etapa 3: Validación de Resultados y Métricas del Dashboard (TP N.º 3 y N.º 4)
Se ejecutan controles de integridad matemática, análisis de valores extremos y verificación de los indicadores ponderados mostrados en el tablero de Looker Studio (*"Más Docentes, Menos Escuelas"*).

In [20]:
# Control 1: Análisis de valor extremo (Outlier) en Daireaux (2025)
daireaux = df[(df['municipio'] == 'daireaux') & (df['anio'] == 2025)]
matricula_d = daireaux['matricula'].values[0]
docentes_d = daireaux['docentes'].values[0]
escuelas_d = daireaux['unidades_de_servicio'].values[0]
ratio_d = daireaux['alumnos_por_docente'].values[0]
ratio_escuela_d = daireaux['alumnos_por_escuela'].values[0]

if round(matricula_d / docentes_d, 2) == ratio_d and round(matricula_d / escuelas_d, 2) == ratio_escuela_d:
    print(f"[OK] Control Daireaux (2025): Ratio {ratio_d} alum/doc confirmado ({matricula_d} alumnos, {docentes_d} docentes en {escuelas_d} escuelas).")
else:
    print("[ERROR] Discrepancia en el cálculo de Daireaux.")

# Control 2: Ausencia de valores nulos o infinitos en métricas derivadas
nulos = df[['alumnos_por_docente', 'alumnos_por_escuela']].isnull().sum().sum()
infinitos = np.isinf(df[['alumnos_por_docente', 'alumnos_por_escuela']]).sum().sum()

if nulos == 0 and infinitos == 0:
    print("[OK] Control de Nulidad: 0 valores nulos y 0 valores infinitos en ratios.")
else:
    print("[ERROR] Existen valores corruptos en los ratios.")

# Control 3: Integridad de totales tras segmentación Conurbano / Interior (2025)
total_original_2025 = df[df['anio'] == 2025]['matricula'].sum()
total_agrupado = df_2025.groupby('zona')['matricula'].sum().sum()

if total_original_2025 == total_agrupado:
    print(f"[OK] Control Territorial (2025): La suma por zonas ({total_agrupado:,}) coincide al 100% con el total provincial.")
else:
    print("[ERROR] Discrepancia en totales territoriales.")

# Control 4: Verificación de KPIs ponderados del Dashboard (Looker Studio - Página 1)
mat_2015, mat_2025 = df[df['anio'] == 2015]['matricula'].sum(), df[df['anio'] == 2025]['matricula'].sum()
doc_2015, doc_2025 = df[df['anio'] == 2015]['docentes'].sum(), df[df['anio'] == 2025]['docentes'].sum()

kpi_ratio_doc_2025 = round(mat_2025 / doc_2025, 2)
kpi_var_doc = round(((doc_2025 - doc_2015) / doc_2015) * 100, 1)
kpi_var_mat = round(((mat_2025 - mat_2015) / mat_2015) * 100, 1)

print(f"[OK] KPIs Dashboard 2025 -> Matrícula: {mat_2025:,} (+{kpi_var_mat}%) | Docentes: {doc_2025:,} (+{kpi_var_doc}%) | Ratio Ponderado: {kpi_ratio_doc_2025}")

[OK] Control Daireaux (2025): Ratio 1.28 alum/doc confirmado (5940 alumnos, 4640 docentes en 61 escuelas).
[OK] Control de Nulidad: 0 valores nulos y 0 valores infinitos en ratios.
[OK] Control Territorial (2025): La suma por zonas (5,001,493) coincide al 100% con el total provincial.
[OK] KPIs Dashboard 2025 -> Matrícula: 5,001,493 (+1.8%) | Docentes: 590,849 (+20.2%) | Ratio Ponderado: 8.46
